# Tecnológico Nacional de México · Instituto Tecnológico de Ciudad Madero
### Departamento de Sistemas y Computación
### Ingeniería en Sistemas Computacionales
---
## Práctica de Laboratorio: Análisis y visualización de calificaciones con Pandas y Matplotlib
**Asignatura:** Tópicos Avanzados de Programación  
**Docente:** Dr. Alejandro Estrada Padilla  
**Estudiante:** Eros Manuel Torres Arzola  


## Importación de librerías
Importamos `pandas`, `numpy` y `matplotlib.pyplot`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## Ejercicio 1. Carga y revisión
Lee `calificaciones_isc.csv` en un DataFrame llamado `df`.
- Las primeras 5 filas.
- El número de filas y de columnas.
- El tipo de dato de cada columna.
- El resumen estadístico de calificación y de asistencias.

> **Nota:** ¿Por qué el archivo tiene 288 registros y no 24?
> Porque son 24 alumnos evaluados en 4 materias distintas, y cada materia cuenta con 3 unidades de evaluación (24 * 4 * 3 = 288 registros).

In [ ]:
df = pd.read_csv('calificaciones_isc.csv')

print('--- Primeras 5 filas (df.head()) ---')
display(df.head())

filas, cols = df.shape
print(f'Dimensiones del DataFrame: {filas} filas x {cols} columnas')

print('\n--- Tipos de datos por columna ---')
print(df.dtypes)

print('\n--- Resumen estadístico (calificacion y asistencias) ---')
display(df[['calificacion', 'asistencias']].describe())

## Ejercicio 2. Filtros
Sin modificar `df`, obtén y cuenta las filas de cada caso:
- Solo el grupo A.
- Unidades con calificación menor a 70.
- Bases de datos, del grupo B, con calificación de al menos 80.

*Sin usar ciclos para contarlas.*

In [ ]:
filtro_grupo_a = df[df['grupo'] == 'A']
print(f'Registros del Grupo A: {len(filtro_grupo_a)}')

filtro_menor_70 = df[df['calificacion'] < 70]
print(f'Unidades con calificación menor a 70: {len(filtro_menor_70)}')

filtro_bd_b_80 = df[(df['materia'] == 'Bases de datos') & (df['grupo'] == 'B') & (df['calificacion'] >= 80)]
print(f'Bases de datos, Grupo B, calificación >= 80: {len(filtro_bd_b_80)}')

## Ejercicio 3. Columna calculada
Crea en `df` la columna `situacion`. Su valor será `"Aprobado"` si calificación es mayor o igual a 70, y `"Reprobado"` en cualquier otro caso sin usar ciclos.

In [ ]:
df['situacion'] = np.where(df['calificacion'] >= 70, 'Aprobado', 'Reprobado')
display(df.head(8))

## Ejercicio 4. Resúmenes y exportación CSV
Calcula e imprime los resúmenes estadísticos solicitados y exporta los promedios de los alumnos a `promedios_alumnos.csv` sin el índice de Pandas.

In [ ]:
print('1. Promedio de calificación por materia:')
promedio_materia = df.groupby('materia')['calificacion'].mean().round(2)
print(promedio_materia)

print('\n2. Promedio por grupo y materia:')
promedio_grupo_materia = df.groupby(['grupo', 'materia'])['calificacion'].mean().round(2)
print(promedio_grupo_materia)

print('\n3. Promedio general de cada alumno (ordenado descendentemente):')
promedios_alumnos = (
    df.groupby(['matricula', 'nombre'])['calificacion']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'calificacion': 'promedio'})
    .sort_values(by='promedio', ascending=False)
)
display(promedios_alumnos)

print('\n4. Promedio general por unidad:')
promedio_unidad = df.groupby('unidad')['calificacion'].mean().round(2)
print(promedio_unidad)

print('\n5. Conteo de unidades aprobadas y reprobadas:')
print(df['situacion'].value_counts())

# Exportar a CSV conservando matrícula y nombre, sin índice de Pandas
promedios_alumnos.to_csv('promedios_alumnos.csv', index=False)
print("\nArchivo 'promedios_alumnos.csv' exportado exitosamente.")

## Ejercicio 5. Gráficas con Matplotlib
Se generan las tres figuras solicitadas con título y nombres de ejes, guardándolas como PNG antes de mostrarlas.

In [ ]:
# Gráfica 1: Barras del promedio por materia (eje Y de 0 a 100)
fig1, ax1 = plt.subplots(figsize=(8, 5))
colores = ['#2b5c8f', '#d95f02', '#7570b3', '#1b9e77']
barras = ax1.bar(promedio_materia.index, promedio_materia.values, color=colores, width=0.55, edgecolor='#222222', zorder=3)
ax1.set_ylim(0, 100)
ax1.set_title('Promedio de Calificación por Materia', fontsize=14, fontweight='bold', pad=12)
ax1.set_xlabel('Materia', fontsize=11, fontweight='bold')
ax1.set_ylabel('Calificación Promedio (0 - 100)', fontsize=11, fontweight='bold')
ax1.grid(axis='y', linestyle='--', alpha=0.6, zorder=0)
for b in barras:
    h = b.get_height()
    ax1.annotate(f'{h:.2f}', xy=(b.get_x() + b.get_width() / 2, h), xytext=(0, 4), textcoords='offset points', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.axhline(70, color='#d9534f', linestyle=':', linewidth=1.5, label='Mínimo aprobatorio (70)')
ax1.legend(loc='upper right')
plt.tight_layout()
plt.savefig('promedio_materias.png', dpi=300)
plt.show()

In [ ]:
# Gráfica 2: Histograma de todas las calificaciones con línea vertical en 70
fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.hist(df['calificacion'], bins=10, color='#3470a3', edgecolor='white', alpha=0.85, zorder=3)
ax2.axvline(70, color='#e74c3c', linestyle='--', linewidth=2, label='Límite aprobatorio (70)')
ax2.set_title('Distribución de Todas las Calificaciones Registradas', fontsize=14, fontweight='bold', pad=12)
ax2.set_xlabel('Calificación Obtenida', fontsize=11, fontweight='bold')
ax2.set_ylabel('Frecuencia (Número de registros)', fontsize=11, fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.6, zorder=0)
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig('histograma.png', dpi=300)
plt.show()

In [ ]:
# Gráfica 3: Líneas del promedio por unidad (una línea por materia)
fig3, ax3 = plt.subplots(figsize=(9, 5.5))
promedio_materia_unidad = df.groupby(['unidad', 'materia'])['calificacion'].mean().unstack()
marcadores = ['o', 's', '^', 'D']
colores_lineas = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for idx, col in enumerate(promedio_materia_unidad.columns):
    ax3.plot(promedio_materia_unidad.index, promedio_materia_unidad[col],
             marker=marcadores[idx % len(marcadores)],
             color=colores_lineas[idx % len(colores_lineas)],
             linewidth=2.2, markersize=7, label=col, zorder=3)
    for u in promedio_materia_unidad.index:
        val = promedio_materia_unidad.loc[u, col]
        ax3.annotate(f'{val:.1f}', (u, val), textcoords='offset points', xytext=(0, 7), ha='center', fontsize=8.5)
ax3.set_title('Tendencia del Promedio por Unidad según Materia', fontsize=14, fontweight='bold', pad=12)
ax3.set_xlabel('Unidad de Evaluación', fontsize=11, fontweight='bold')
ax3.set_ylabel('Calificación Promedio', fontsize=11, fontweight='bold')
ax3.set_xticks([1, 2, 3])
ax3.set_xticklabels(['Unidad 1', 'Unidad 2', 'Unidad 3'])
ax3.grid(True, linestyle='--', alpha=0.6, zorder=0)
ax3.axhline(70, color='gray', linestyle=':', linewidth=1, label='Aprobatorio (70)')
ax3.legend(title='Materias', loc='best')
plt.tight_layout()
plt.savefig('tendencia_unidades.png', dpi=300)
plt.show()

## Ejercicio 6. Lectura de resultados e interpretación

### 1. ¿Qué materia quedó con el promedio más bajo?
**Respuesta:** La materia con el promedio más bajo es **Matemáticas discretas** con un promedio de **66.08** (la única materia por debajo de 70). En comparación, Redes obtuvo 76.03, Bases de datos 73.85 y Programación 72.97.

### 2. ¿El promedio general de la unidad 3 es menor que el de las unidades 1 y 2?
**Respuesta:** **Sí.** El promedio de la Unidad 3 es de **69.79**, siendo menor al de la Unidad 1 (**73.75**) y al de la Unidad 2 (**73.16**). Se aprecia una caída de rendimiento hacia el final del periodo en materias como Programación y Matemáticas discretas.

### 3. ¿Cuántos alumnos quedan bajo 70 en su promedio general?
**Respuesta:** Quedan exactamente **7 alumnos** (de 24) con un promedio general menor a 70:
1. Ana López (69.92)
2. Iván Ramírez (69.92)
3. Fernanda Rodríguez (69.50)
4. Diana Hernández (69.08)
5. Gabriel Sánchez (67.92)
6. Víctor Ramos (66.17)
7. Kevin Gómez (64.08)